# 02 — RVC v2 Fine-Tuning
**Neural Voice Identity Control & Deepfake Audio Analysis — Disertație 2026**

Acest notebook fine-tunează RVC v2 pentru fiecare din cele 3 voci.

**Prerequisite:** Rulează mai întâi `01_data_collection.ipynb` și asigură-te că ai clipuri în `data/{voice_id}/clips/`.

**Timp estimat:** ~1-2h per voce pe Colab T4 (cu 15-30 min audio curat).

---
> ⚡ **Asigură-te că runtime-ul este setat pe GPU:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Verificare GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠ WARNING: No GPU detected. Training will be very slow.')
    print('Go to: Runtime → Change runtime type → GPU → T4')

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone RVC v2 repository
import os

RVC_DIR = '/content/RVC'

if not os.path.exists(RVC_DIR):
    print('Cloning RVC repository...')
    !git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git {RVC_DIR}
else:
    print('RVC already cloned.')
    !git -C {RVC_DIR} pull

os.chdir(RVC_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# Instalare dependente RVC
import subprocess, sys

def install(pkg, extra=[]):
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg] + extra,
        capture_output=True, text=True
    )
    status = 'OK' if r.returncode == 0 else 'FAIL'
    print(f'  [{status}] {pkg}')
    return r.returncode == 0

print('Instalare dependente RVC...')
for pkg in [
    'numpy', 'scipy', 'librosa==0.9.2', 'soundfile',
    'praat-parselmouth>=0.4.2', 'torchcrepe',
    'faiss-cpu', 'ffmpeg-python', 'av',
    'tensorboard', 'transformers', 'huggingface_hub',
    'gradio', 'numba', 'einops', 'local_attention',
]:
    install(pkg)

if not install('pyworld'):
    install('pyworld', ['--no-build-isolation'])

print('\nGata!')

In [ ]:
# Download required pre-trained models (HuBERT, RMVPE, pretrained RVC v2)
import os

os.makedirs(f'{RVC_DIR}/assets/pretrained_v2', exist_ok=True)
os.makedirs(f'{RVC_DIR}/assets/uvr5_weights', exist_ok=True)

BASE_MODELS_URL = 'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main'

files_to_download = [
    (f'{BASE_MODELS_URL}/pretrained_v2/f0D48k.pth',  f'{RVC_DIR}/assets/pretrained_v2/f0D48k.pth'),
    (f'{BASE_MODELS_URL}/pretrained_v2/f0G48k.pth',  f'{RVC_DIR}/assets/pretrained_v2/f0G48k.pth'),
    (f'{BASE_MODELS_URL}/pretrained_v2/f0D40k.pth',  f'{RVC_DIR}/assets/pretrained_v2/f0D40k.pth'),
    (f'{BASE_MODELS_URL}/pretrained_v2/f0G40k.pth',  f'{RVC_DIR}/assets/pretrained_v2/f0G40k.pth'),
    (f'{BASE_MODELS_URL}/hubert_base.pt',             f'{RVC_DIR}/assets/hubert/hubert_base.pt'),
    (f'{BASE_MODELS_URL}/rmvpe.pt',                   f'{RVC_DIR}/assets/rmvpe/rmvpe.pt'),
]

for url, dest in files_to_download:
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        print(f'Downloading: {os.path.basename(dest)}')
        !wget -q '{url}' -O '{dest}'
    else:
        print(f'Already exists: {os.path.basename(dest)}')

print('\n✓ Pre-trained models ready.')

## Configurare antrenare

**TODO:** Completează configurarea de mai jos cu ID-urile vocilor tale.

In [ ]:
import os

# ============================================================
# CONFIGURARE
# ============================================================
DRIVE_BASE = '/content/drive/MyDrive/disertatie'
CLIPS_BASE = f'{DRIVE_BASE}/data/processed'
MODELS_OUT = f'{DRIVE_BASE}/models'
os.makedirs(MODELS_OUT, exist_ok=True)

VOICES_TO_TRAIN = [
    {
        'voice_id': 'voice_1',
        'display_name': 'Călin Georgescu',
        'sample_rate': 40000,
        'epochs': 200,
        'batch_size': 8,
        'save_every_epoch': 50,
    },
    {
        'voice_id': 'voice_2',
        'display_name': 'Nicușor Dan',
        'sample_rate': 40000,
        'epochs': 200,
        'batch_size': 8,
        'save_every_epoch': 50,
    },
    {
        'voice_id': 'voice_3',
        'display_name': 'Diana Șoșoacă',
        'sample_rate': 40000,
        'epochs': 200,
        'batch_size': 8,
        'save_every_epoch': 50,
    },
]
# ============================================================

print('Training configuration:')
for v in VOICES_TO_TRAIN:
    clips_dir = f"{CLIPS_BASE}/{v['voice_id']}/clips"
    clip_count = len([f for f in os.listdir(clips_dir) if f.endswith('.wav')]) if os.path.exists(clips_dir) else 0
    print(f"  {v['display_name']}: {clip_count} clips")

In [ ]:
import subprocess, shutil, sys, threading, time, glob, os, json
import numpy as np
import soundfile as sf
import torch
sys.path.insert(0, RVC_DIR)

def _create_config(log_dir, sr, batch_size):
    config = {
        "train": {
            "log_interval": 200, "seed": 1234, "epochs": 20000,
            "learning_rate": 1e-4, "betas": [0.8, 0.99], "eps": 1e-9,
            "batch_size": batch_size, "fp16_run": True, "lr_decay": 0.999875,
            "segment_size": 12800, "init_lr_ratio": 1, "warmup_epochs": 0,
            "c_mel": 45, "c_kl": 1.0
        },
        "data": {
            "max_wav_value": 32768.0, "sampling_rate": sr,
            "filter_length": 2048, "hop_length": sr // 100,
            "win_length": 2048, "n_mel_channels": 125,
            "mel_fmin": 0.0, "mel_fmax": None
        },
        "model": {
            "inter_channels": 192, "hidden_channels": 192,
            "filter_channels": 768, "n_heads": 2, "n_layers": 6,
            "kernel_size": 3, "p_dropout": 0, "resblock": "1",
            "resblock_kernel_sizes": [3, 7, 11],
            "resblock_dilation_sizes": [[1,3,5],[1,3,5],[1,3,5]],
            "upsample_rates": [10, 10, 2, 2],
            "upsample_initial_channel": 512,
            "upsample_kernel_sizes": [16, 16, 4, 4],
            "use_spectral_norm": False, "gin_channels": 256, "spk_embed_dim": 109
        }
    }
    with open(os.path.join(log_dir, 'config.json'), 'w') as f:
        json.dump(config, f, indent=2)
    print(f'  config.json creat')

def _extract_hubert_features(log_dir):
    from transformers import HubertModel, Wav2Vec2FeatureExtractor
    wav_dir = os.path.join(log_dir, '1_16k_wavs')
    out_dir = os.path.join(log_dir, '3_feature768')
    os.makedirs(out_dir, exist_ok=True)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    is_half = torch.cuda.is_available()
    print(f'  Incarcare HuBERT (device={device})...')
    extractor = Wav2Vec2FeatureExtractor.from_pretrained('facebook/hubert-base-ls960')
    model = HubertModel.from_pretrained('facebook/hubert-base-ls960').to(device)
    if is_half:
        model = model.half()
    model.eval()

    wav_files = sorted(f for f in os.listdir(wav_dir) if f.endswith('.wav'))
    print(f'  Procesare {len(wav_files)} fisiere...')
    for i, fname in enumerate(wav_files):
        out_path = os.path.join(out_dir, fname.replace('.wav', '.npy'))
        if os.path.exists(out_path):
            continue
        audio, _ = sf.read(os.path.join(wav_dir, fname))
        audio = audio.astype(np.float32)
        if len(audio.shape) > 1:
            audio = audio.mean(axis=1)
        inputs = extractor(audio, sampling_rate=16000, return_tensors='pt', padding=True)
        with torch.no_grad():
            vals = inputs.input_values.to(device)
            if is_half:
                vals = vals.half()
            out = model(vals, output_hidden_states=True)
            feats = out.hidden_states[9][0].float().cpu().numpy()
        np.save(out_path, feats)
        if (i + 1) % 50 == 0 or i == len(wav_files) - 1:
            print(f'  [{i+1}/{len(wav_files)}] done')
    print(f'  HuBERT features gata.')

def _create_filelist(log_dir):
    gt_wavs = sorted(glob.glob(f'{log_dir}/0_gt_wavs/*.wav'))
    lines = []
    skipped = 0
    for wav_path in gt_wavs:
        base = os.path.basename(wav_path)
        name_no_ext = base[:-4]
        f0_path    = f'{log_dir}/2a_f0/{base}.npy'
        f0nsf_path = f'{log_dir}/2b-f0nsf/{base}.npy'
        feat_path  = f'{log_dir}/3_feature768/{name_no_ext}.npy'
        if not all(os.path.exists(p) for p in [f0_path, f0nsf_path, feat_path]):
            skipped += 1
            continue
        lines.append(f'{wav_path}|{f0_path}|{f0nsf_path}|{feat_path}|0')
    with open(f'{log_dir}/filelist.txt', 'w') as f:
        f.write('\n'.join(lines))
    print(f'  filelist.txt: {len(lines)} intrari ({skipped} sarite)')

def _watch_checkpoints(log_dir, models_out, voice_id, stop_event):
    copied = set()
    while not stop_event.is_set():
        for ckpt in glob.glob(f'{log_dir}/*.pth'):
            if ckpt not in copied:
                dest = f'{models_out}/{voice_id}_ckpt_{os.path.basename(ckpt)}'
                try:
                    shutil.copy2(ckpt, dest)
                    copied.add(ckpt)
                    print(f'  [Drive] Checkpoint salvat: {os.path.basename(dest)}')
                except Exception:
                    pass
        time.sleep(60)

def train_voice(voice_config):
    voice_id      = voice_config['voice_id']
    name          = voice_config['display_name']
    sr            = voice_config['sample_rate']
    epochs        = voice_config['epochs']
    batch_size    = voice_config['batch_size']
    save_interval = voice_config['save_every_epoch']
    clips_dir     = f"{CLIPS_BASE}/{voice_id}/clips"
    log_dir       = f'{RVC_DIR}/logs/{voice_id}'
    os.makedirs(log_dir, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'Training: {name} ({voice_id})')
    print(f'Epochs: {epochs} | Batch: {batch_size}')
    print(f'{"="*60}')

    _create_config(log_dir, sr, batch_size)

    stop_event = threading.Event()
    watcher = threading.Thread(
        target=_watch_checkpoints,
        args=(log_dir, MODELS_OUT, voice_id, stop_event),
        daemon=True
    )
    watcher.start()

    try:
        print('\n[1/4] Preprocessing...')
        subprocess.run([
            'python', f'{RVC_DIR}/infer/modules/train/preprocess.py',
            clips_dir, str(sr), '2', log_dir, 'False', '3.7',
        ], check=True, cwd=RVC_DIR)

        print('\n[2/4] Extractare features HuBERT...')
        _extract_hubert_features(log_dir)

        print('\n[3/4] Extractare F0 rmvpe...')
        subprocess.run([
            'python', f'{RVC_DIR}/infer/modules/train/extract/extract_f0_print.py',
            log_dir, '2', 'rmvpe',
        ], check=True, cwd=RVC_DIR)

        print('\n[3.5/4] Creare filelist antrenare...')
        _create_filelist(log_dir)

        print(f'\n[4/4] Antrenare {epochs} epoci...')
        subprocess.run([
            'python', f'{RVC_DIR}/infer/modules/train/train.py',
            '-e', voice_id, '-sr', f'{sr//1000}k', '-f0', '1',
            '-bs', str(batch_size), '-g', '0',
            '-te', str(epochs), '-se', str(save_interval),
            '-pg', f'{RVC_DIR}/assets/pretrained_v2/f0G{sr//1000}k.pth',
            '-pd', f'{RVC_DIR}/assets/pretrained_v2/f0D{sr//1000}k.pth',
            '-l', '0', '-c', '0', '-sw', '0', '-v', 'v2',
        ], check=True, cwd=RVC_DIR)

    finally:
        stop_event.set()
        watcher.join(timeout=5)

    checkpoints = sorted(glob.glob(f'{log_dir}/*.pth'))
    if checkpoints:
        dest = f'{MODELS_OUT}/{voice_id}.pth'
        shutil.copy2(checkpoints[-1], dest)
        print(f'\n[Drive] Model final: {dest}')
    else:
        print('\nWARNING: Nu s-a gasit checkpoint final.')

print('Functiile train_voice si _create_filelist definite.')

In [ ]:
import faiss
import numpy as np

def build_index(voice_id):
    log_dir     = f'{RVC_DIR}/logs/{voice_id}'
    feature_dir = f'{log_dir}/3_feature768'

    if not os.path.exists(feature_dir):
        print(f'Feature dir negasit: {feature_dir}')
        return

    npy_files = sorted(glob.glob(f'{feature_dir}/*.npy'))
    if not npy_files:
        print('Nu exista fisiere de features.')
        return

    features = np.concatenate([np.load(f) for f in npy_files], axis=0)
    print(f'Features incarcate: {features.shape}')

    n_ivf = min(int(16 * (features.shape[0] ** 0.5)), features.shape[0])
    index = faiss.index_factory(768, f'IVF{n_ivf},Flat')
    index.train(features)
    index.add(features)

    index_path = f'{log_dir}/{voice_id}_v2.index'
    faiss.write_index(index, index_path)

    dest = f'{MODELS_OUT}/{voice_id}.index'
    shutil.copy2(index_path, dest)
    print(f'Index salvat: {dest}')

print('Functia build_index definita.')

In [ ]:
# ============================================================
# ANTRENARE — RULEAZĂ CELULA ASTA
# Durată estimată: ~1-2h per voce pe T4
# ============================================================

# Antrenează secvențial toate vocile
# Pentru a antrena doar una, comentează celelalte din VOICES_TO_TRAIN

for voice_config in VOICES_TO_TRAIN:
    try:
        train_voice(voice_config)
        build_index(voice_config['voice_id'])
        print(f"\n✓✓ DONE: {voice_config['display_name']}")
    except Exception as e:
        print(f"\n✗ ERROR for {voice_config['voice_id']}: {e}")
        import traceback
        traceback.print_exc()

print('\n' + '='*60)
print('Fine-tuning complete. Continuă cu: 03_main_app.ipynb')
print('='*60)

In [ ]:
# Test rapid de inferență — verificare că modelul funcționează
import sys
import soundfile as sf
import numpy as np

sys.path.insert(0, RVC_DIR)

TEST_VOICE_ID = 'voice_1'  # TODO: schimbă cu vocea pe care vrei să o testezi
TEST_INPUT = None  # TODO: calea către un .wav de test (sau lasă None pentru test sintetic)

if TEST_INPUT is None:
    # Generează un semnal de test sintetic
    sr = 40000
    t = np.linspace(0, 2, 2 * sr)
    test_audio = (np.sin(2 * np.pi * 200 * t) * 0.3).astype(np.float32)
    TEST_INPUT = '/tmp/test_input.wav'
    sf.write(TEST_INPUT, test_audio, sr)
    print('Created synthetic test audio.')

model_path = f'{MODELS_OUT}/{TEST_VOICE_ID}.pth'
index_path = f'{MODELS_OUT}/{TEST_VOICE_ID}.index'

if not os.path.exists(model_path):
    print(f'Model not found: {model_path}')
    print('Fine-tuning must complete first.')
else:
    from infer.modules.vc.modules import VC
    from configs.config import Config
    
    config = Config()
    vc = VC(config)
    vc.get_vc(model_path)
    
    tgt_sr, output = vc.vc_single(
        sid=0,
        input_audio_path=TEST_INPUT,
        f0_up_key=0,
        f0_file=None,
        f0_method='rmvpe',
        file_index=index_path if os.path.exists(index_path) else '',
        index_rate=0.75,
        filter_radius=3,
        resample_sr=0,
        rms_mix_rate=0.25,
        protect=0.33,
    )
    
    sf.write('/tmp/test_output.wav', output, tgt_sr)
    print(f'\n✓ Test inference successful!')
    print(f'Output sample rate: {tgt_sr}')
    print(f'Output length: {len(output)/tgt_sr:.2f}s')
    
    from IPython.display import Audio, display
    print('Input audio:')
    display(Audio(TEST_INPUT))
    print('Converted audio:')
    display(Audio('/tmp/test_output.wav'))